# 怎样独立完成可复现、可交接的预测项目？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

原记录 17379 缺失小时 165


最差的五个小时                      target  prediction  absolute_error
2012-10-02 08:00:00   134.0  779.359585      645.359585
2012-09-18 17:00:00   338.0  857.571973      519.571973
2012-11-23 08:00:00    94.0  601.184708      507.184708
2012-09-18 18:00:00   281.0  774.339745      493.339745
2012-12-26 08:00:00    31.0  502.179592      471.179592


{
  "raw_rows": 17379,
  "usable_rows": 16950,
  "test_rows": 2543,
  "selected_model": "hist_gradient_boosting",
  "validation_mae": {
    "ridge": 58.90707101568961,
    "hist_gradient_boosting": 53.68451649707337
  },
  "test_mae": 55.29867861252311,
  "yesterday_baseline_mae": 77.56940621313409,
  "test_rmse": 86.64522709073388,
  "relative_mae_change": 0.2871071043062862,
  "api_status": 200,
  "invalid_api_status": 422,
  "leakage_columns_excluded": true,
  "elapsed_seconds": 1.407
}


## 1. 补全：检查时间与泄漏
核对禁止列不在输入中，随机挑一个测试时间检查lag24确实来自前24小时。

In [2]:
assert not set(features) & forbidden
t = test.index[20]
assert np.isclose(test.loc[t, 'lag24'], hourly.loc[t - pd.Timedelta(hours=24), 'cnt'])
assert train.index.max() < validation.index.min() and validation.index.max() < test.index.min()
print('时间与特征检查通过')

时间与特征检查通过


## 2. 修改：按天定位错误
将全部测试误差按日期聚合，打印误差最大的五天；不能只挑好看的168小时展示。

In [3]:
daily = result['absolute_error'].groupby(result.index.date).mean().sort_values(ascending=False)
assert len(daily) > 5 and np.isclose(result['absolute_error'].mean(), model_mae)
print(daily.head())

2012-12-24    150.780337
2012-12-26    141.071077
2012-09-18    132.327717
2012-12-25    123.897570
2012-11-24    120.153803
Name: absolute_error, dtype: float64


## 3. 迁移：把预测变成人员安排
假设一名调度员可处理80次需求，按预测向上取整配置；与按训练平均需求配置对照，报告工时和未覆盖需求。成本与能力是假设。

In [4]:
capacity = 80
staff = np.maximum(1, np.ceil(predicted / capacity)).astype(int)
fixed = int(max(1, np.ceil(train['target'].mean() / capacity)))
shortage = np.maximum(0, test['target'].to_numpy() - staff * capacity)
fixed_shortage = np.maximum(0, test['target'].to_numpy() - fixed * capacity)
assert len(staff) == len(test) and np.all(staff >= 1) and np.all(shortage >= 0)
print({'forecast_staff_hours': int(staff.sum()), 'fixed_staff_hours': fixed * len(test), 'forecast_uncovered': float(shortage.sum()), 'fixed_uncovered': float(fixed_shortage.sum())})

{'forecast_staff_hours': 9086, 'fixed_staff_hours': 7629, 'forecast_uncovered': 34695.0, 'fixed_uncovered': 217235.0}
